# 07 - Stage 1 Viterbi Smoothing

This notebook learns a stage transition matrix from existing coded conversations and applies Viterbi decoding to per-message predicted probabilities. The ordered stage codes define the expected progression 200 -> 600 -> 900, but observed codes remain the source labels and are never overwritten.

Viterbi smoothing is evaluated against the existing fold predictions and source codes, grouped by chat.

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.model_selection import StratifiedGroupKFold

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

WORKSPACE = Path.cwd()
PROCESSED_DIR = WORKSPACE / "data" / "processed"
MODEL_DIR = WORKSPACE / "models"
REPORT_DIR = WORKSPACE / "reports"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

messages = pd.read_csv(PROCESSED_DIR / "coded_stage_messages.csv", dtype={"category": "string"})
messages["text"] = messages["text"].fillna("")
messages["category"] = messages["category"].astype(str)
messages = messages.sort_values(["chat_id", "msg_idx"]).copy()
messages["previous_text"] = messages.groupby("chat_id")["text"].shift(1).fillna("")
messages["next_text"] = messages.groupby("chat_id")["text"].shift(-1).fillna("")
messages["model_text"] = (
    "previous: " + messages["previous_text"] + " target: " + messages["text"] + " next: " + messages["next_text"]
)

CLASSES = ["200", "600", "900"]
CLASS_INDEX = {stage: index for index, stage in enumerate(CLASSES)}
assert messages["category"].isin(CLASSES).all()
assert not messages.duplicated(["chat_id", "msg_idx"]).any()


## Learn transitions from existing codes

The 3x3 matrix is smoothed with a small pseudo-count so unseen transitions remain finite. Reversals and unusual jumps are visible rather than silently removed.

In [2]:
def learn_transition_matrix(frame: pd.DataFrame, alpha: float = 1e-3) -> np.ndarray:
    counts = np.full((len(CLASSES), len(CLASSES)), alpha, dtype=float)
    for _, chat in frame.groupby("chat_id", sort=False):
        sequence = chat.sort_values("msg_idx")["category"].tolist()
        for previous, current in zip(sequence[:-1], sequence[1:]):
            counts[CLASS_INDEX[previous], CLASS_INDEX[current]] += 1.0
    counts /= counts.sum(axis=1, keepdims=True)
    return counts


transitions = learn_transition_matrix(messages)
transition_table = pd.DataFrame(transitions, index=CLASSES, columns=CLASSES)
display(transition_table)
np.save(PROCESSED_DIR / "stage1_transition_matrix.npy", transitions)
pd.DataFrame(transitions, index=CLASSES, columns=CLASSES).to_csv(REPORT_DIR / "stage1_transition_matrix.csv")

,200,600,900
200,0.594379,0.214210,0.191411
600,0.077773,0.795792,0.126434
900,0.085585,0.134114,0.780301


## Viterbi decoding

Emission probabilities come from a model fitted in each outer fold. The implementation uses log probabilities to avoid numerical underflow.

In [ ]:
def viterbi_decode(emission_probabilities: np.ndarray, transition: np.ndarray) -> np.ndarray:
    epsilon = np.finfo(float).eps
    emission = np.log(np.clip(emission_probabilities, epsilon, 1.0))
    log_transition = np.log(np.clip(transition, epsilon, 1.0))
    length, number_of_states = emission.shape
    if length == 0:
        return np.empty(0, dtype=int)
    scores = np.full((length, number_of_states), -np.inf)
    backpointers = np.zeros((length, number_of_states), dtype=int)
    scores[0] = emission[0]
    for position in range(1, length):
        candidates = scores[position - 1][:, None] + log_transition
        backpointers[position] = np.argmax(candidates, axis=0)
        scores[position] = emission[position] + candidates.max(axis=0)
    path = np.zeros(length, dtype=int)
    path[-1] = np.argmax(scores[-1])
    for position in range(length - 1, 0, -1):
        path[position - 1] = backpointers[position, path[position]]
    return path


def build_model() -> Pipeline:
    features = FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 3), min_df=2, sublinear_tf=True)),
        ("character", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
    ])
    classifier = LogisticRegression(
        class_weight="balanced",
        solver="lbfgs",
        max_iter=2000,
        random_state=RANDOM_SEED,
    )
    return Pipeline([("features", features), ("classifier", classifier)])


X = messages["model_text"].to_numpy()
y = messages["category"].to_numpy()
groups = messages["chat_id"].to_numpy()
n_splits = min(5, messages["chat_id"].nunique())
evaluation_rows = []
decoded_rows = []

for repeat in range(3):
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED + repeat)
    for fold, (train_index, test_index) in enumerate(splitter.split(X, y, groups)):
        model = build_model()
        model.fit(X[train_index], y[train_index])
        emission_probabilities = model.predict_proba(X[test_index])
        model_classes = [str(category) for category in model.classes_]

        test_frame = messages.iloc[test_index].copy().reset_index(drop=True)
        for chat_id, chat_positions in test_frame.groupby("chat_id", sort=False).groups.items():
            positions = np.asarray(sorted(chat_positions), dtype=int)
            emission = emission_probabilities[positions]
            emission = np.column_stack([emission[:, model_classes.index(stage)] for stage in CLASSES])
            decoded_path = viterbi_decode(emission, transitions)
            decoded_labels = np.asarray(CLASSES)[decoded_path]

            local_truth = test_frame.loc[positions, "category"].to_numpy()
            local_unsmoothed = np.asarray(model_classes)[np.argmax(emission_probabilities[positions], axis=1)]
            for local_index, source_position in enumerate(positions):
                decoded_rows.append({
                    "repeat": repeat,
                    "fold": fold,
                    "chat_id": chat_id,
                    "source_row_position": int(source_position),
                    "true_category": local_truth[local_index],
                    "unsmoothed_category": local_unsmoothed[local_index],
                    "viterbi_category": decoded_labels[local_index],
                })

decoded = pd.DataFrame(decoded_rows)
unsmoothed = precision_recall_fscore_support(decoded["true_category"], decoded["unsmoothed_category"], labels=CLASSES, average="macro", zero_division=0)
smoothed = precision_recall_fscore_support(decoded["true_category"], decoded["viterbi_category"], labels=CLASSES, average="macro", zero_division=0)
comparison = pd.DataFrame([
    {"method": "unsmoothed", "macro_precision": unsmoothed[0], "macro_recall": unsmoothed[1], "macro_f1": unsmoothed[2]},
    {"method": "viterbi", "macro_precision": smoothed[0], "macro_recall": smoothed[1], "macro_f1": smoothed[2]},
])

display(comparison)
display(decoded.head())

,method,macro_precision,macro_recall,macro_f1
0,unsmoothed,0.694218,0.692325,0.693178
1,viterbi,0.729756,0.681761,0.696095


,repeat,fold,chat_id,source_row_position,true_category,unsmoothed_category,viterbi_category
0,0,0,Pavlov1234,0,600,200,600
1,0,0,Pavlov1234,1,200,600,600
2,0,0,Pavlov1234,2,200,200,600
3,0,0,Pavlov1234,3,200,600,600
4,0,0,Pavlov1234,4,600,600,600


## Save smoothing artifacts

In [4]:
decoded.to_csv(REPORT_DIR / "stage1_viterbi_predictions.csv", index=False)
comparison.to_csv(REPORT_DIR / "stage1_viterbi_comparison.csv", index=False)

with (REPORT_DIR / "stage1_viterbi_config.json").open("w", encoding="utf-8") as handle:
    json.dump({"random_seed": RANDOM_SEED, "classes": CLASSES, "pseudo_count": 1e-3}, handle, indent=2)

print("Saved Viterbi smoothing predictions and comparison.")

Saved Viterbi smoothing predictions and comparison.
